# synthkit — SageMaker Studio walkthrough

Spec -> plan -> verified render -> exact evaluation -> experiment,
end to end. Every cell up to the Bedrock section is deterministic
and LLM-free (StubBackend), so this notebook doubles as the
integration test it demonstrates. Set `RUN_BEDROCK = True` (and
ensure the execution role has `bedrock:InvokeModel`) for the
realistic-prose sections.

PHI posture: synthkit has NO ingestion path. Distributions are
specified, never fitted. Nothing real ever enters the generator.


In [ ]:
import sys
!{sys.executable} -m pip install -q -e .. 2>/dev/null || \
    {sys.executable} -m pip install -q synthkit

RUN_BEDROCK = False
BEDROCK_MODEL_ID = "anthropic.claude-sonnet-4-6-v1:0"
BEDROCK_REGION = "us-west-2"
CORPUS_DIR = "corpus/sagemaker_run_001"


## 1. The spec — the reviewable contract

The reference vertical: inpatient progress notes with current
medications, follow-ups, hard-buried allergies, and a
discontinued-medication distractor trap. In practice you would
compile a spec from plain English (`synthkit.compiler.compile_spec`)
and REVIEW the JSON before this cell; here we load the library
reference.


In [ ]:
from synthkit.examples import reference_spec, reference_rules

spec = reference_spec(size=20, master_seed=42)
spec.validate()
print(spec.to_json()[:1200])


## 2. Plan — blueprints are the ground truth

Deterministic: same spec + seed = same corpus, byte for byte,
on this notebook, on the Actions runner, forever. Labels precede
the data; there is no annotation step.


In [ ]:
from synthkit.planner import plan_corpus, corpus_stats
import json

blueprints = plan_corpus(spec)
print(json.dumps(corpus_stats(blueprints), indent=2))


## 3. Render (deterministic) — the verified corpus

StubBackend echoes planted content into boring prose: valid by
construction, instant, and the baseline every LLM renderer is
compared against.


In [ ]:
from synthkit.harness import StubBackend
from synthkit.renderer import render_corpus

documents, render_report = render_corpus(spec, blueprints,
                                         StubBackend())
print(render_report.format_text())
print()
print(documents["doc_00000"][:400])


## 4. Evaluate — exact, sliced scoring

The naive regex extractor stands in for the vendor model. Replace
`regex_extract` with an adapter around any API and nothing else
changes. Every number below is exact alignment against planted
truth — recall by element, difficulty, persona, verbosity,
abbreviation, plus the distractor trap rate.


In [ ]:
from synthkit.evaluator import FunctionExtractor, evaluate
from synthkit.examples import regex_extract

report = evaluate(blueprints, documents,
                  FunctionExtractor(regex_extract, "regex-naive"),
                  reference_rules())
print(report.format_text())


## 5. Experiment — a hypothesis, measured

Conditions parse from text and resolve against the report. The
verdict carries its evidence.


In [ ]:
from synthkit.harness import Condition, Experiment, run_experiment

exp = Experiment(
    name="allergy bar + discontinued-med trap",
    spec=spec,
    extractor=FunctionExtractor(regex_extract, "regex-naive"),
    conditions=[
        Condition.parse("elements.allergy_flag.recall >= 0.85"),
        Condition.parse(
            "distractors.discontinued_medication.fp_rate <= 0.10"),
    ],
    rules=reference_rules(),
)
result = run_experiment(exp)
print(result.finding())


## 6. Persist — the auditable run artifact

manifest.json carries the verbatim spec, seeds, backend, and
sha256 of every file; a tampered corpus refuses to load. Sync the
run directory to S3 to hand a colleague a reproducible object.


In [ ]:
from pathlib import Path
from synthkit.corpus_io import load_corpus, write_corpus

run_dir = write_corpus(Path(CORPUS_DIR), spec, blueprints,
                       documents, render_report,
                       backend_name="stub")
spec2, bps2, docs2, manifest = load_corpus(run_dir)
assert docs2 == documents and spec2.to_json() == spec.to_json()
print("corpus verified ->", run_dir)
# import subprocess
# subprocess.run(["aws", "s3", "sync", str(run_dir),
#                 "s3://YOUR-BUCKET/synthkit/" + run_dir.name])


## 7. Bedrock — realistic prose (gated)

Same pipeline, real language model. The verifier and fallback
guarantee the corpus renders regardless; the render report tells
you the model's compliance rate. Requires `bedrock:InvokeModel`
on the execution role and the model enabled in the region.


In [ ]:
if RUN_BEDROCK:
    from synthkit.compiler import BedrockBackend
    backend = BedrockBackend(model_id=BEDROCK_MODEL_ID,
                             region=BEDROCK_REGION)
    live_docs, live_render = render_corpus(spec, blueprints,
                                           backend)
    print(live_render.format_text())
    live_report = evaluate(
        blueprints, live_docs,
        FunctionExtractor(regex_extract, "regex-naive"),
        reference_rules())
    print(live_report.format_text())
    write_corpus(Path(CORPUS_DIR + "_bedrock"), spec,
                 blueprints, live_docs, live_render,
                 backend_name="bedrock/" + BEDROCK_MODEL_ID)
else:
    print("RUN_BEDROCK is False — skipped.")


## 8. CI hook

The Actions workflow runs the full smoke net (65 checks, zero
credentials) on every push, plus this same stub pipeline as an
integration job. The Bedrock job is `workflow_dispatch`-gated
for the AWS-controlled runner.
